# Aula 13 · Criando e agendando jobs

## O que é
Existem dois jeitos de acordar alguém:
- ⏰ **Despertador:** toca na **hora marcada**, aconteça o que acontecer.
- 🔔 **Campainha:** toca quando **algo acontece**, quando alguém chega.

Um job pode ser disparado das duas formas:

| Tipo de disparo | Analogia | Quando usar |
|---|---|---|
| **Manual** (Run now) | Você mesmo bate na porta | Testes e reprocessamentos pontuais |
| **Agendado** (intervalo ou cron) | ⏰ Despertador | Dados que chegam em horário previsível |
| **Chegada de arquivo** (file arrival) | 🔔 Campainha do arquivo | Arquivos que chegam sem hora certa |
| **Atualização de tabela** (table update) | 🔔 Campainha da tabela | Quando outro time/job atualiza uma tabela que você usa |
| **Contínuo** (continuous) | Porta sempre aberta | Dados quase em tempo real; o job reinicia assim que termina |

### Agendamento com cron
O cron diz "quando" em 6 campos (o Databricks usa o formato Quartz):

```
segundo  minuto  hora  dia_do_mês  mês  dia_da_semana
   0       0      6        *        *        ?          → todo dia às 06:00
   0      30      8        ?        *      MON-FRI      → dias úteis às 08:30
   0       0     */4       *        *        ?          → a cada 4 horas
```

> Sempre escolha o **fuso horário** (`America/Sao_Paulo`). Sem isso, o job pode rodar 3 horas antes ou depois do esperado.

## Por que importa
Dados chegam em ritmos diferentes. A escolha errada de disparo gera **atraso na entrega** (o dado chegou e ninguém processou) ou **custo desnecessário** (o job roda de hora em hora sem nada novo).

## Como aplicar: três cenários de negócio
Para cada cenário, pare e decida o disparo antes de ler a resposta.

**Cenário 1.** A diretoria olha o relatório de inadimplência **todo dia às 9h**. O sistema de origem fecha o dia às 2h.
> ✅ **Agendado**, cron `0 0 6 * * ?` em `America/Sao_Paulo`. Roda depois do fechamento, com folga antes das 9h.

**Cenário 2.** Um parceiro envia o arquivo de novos empréstimos **algumas vezes por dia, sem horário fixo**.
> ✅ **Chegada de arquivo** na pasta `landing/loans/`. Processa assim que o arquivo chega, sem rodar à toa.

**Cenário 3.** O time de risco mantém a tabela `silver.customers` atualizada por **outro job**, e o nosso relatório deve ser recalculado sempre que ela mudar.
> ✅ **Atualização de tabela** em `growdev.silver.customers`. O nosso job "escuta" a tabela do outro time.

### Onde configurar na interface
No job → painel da direita → **Schedules & Triggers** → **Add trigger** → **Trigger type**: *Scheduled*, *File arrival*, *Table update* ou *Continuous*.

## Checklist da aula
- [ ] Sei a diferença entre despertador (agendado) e campainha (evento)
- [ ] Sei ler uma expressão cron simples
- [ ] Escolhi o disparo certo para os três cenários
- [ ] Sei onde fica **Schedules & Triggers** no job